## Shared Agent Context with Azure CosmosDB

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-ai-projects==2.7.0 azure-identity python-dotenv

### Setting up the Environment Variables

In [ ]:
import os 
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

# loading the cosmosdb environment variables from the .env file
endpoint = os.getenv("COSMOSDB_ENDPOINT")
key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME")

# loading the azure openai environment variables from the .env file
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient

client = CosmosClient(endpoint, key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Load the Patient JSON Data

In [ ]:
import json

with open(
    "patients.json",
    "r"
) as file:

    patient_records = (
        json.load(file)
    )


print(
    len(patient_records),
    "patient records loaded"
)

### Store Patient Data in CosmosDB

In [ ]:
for patient in patient_records:

    container.upsert_item(
        patient
    )

    print(
        "Stored:",
        patient["patientId"]
    )

### Create the CosmosDB Function for our Agent

In [ ]:
def get_patient_context(
    patient_id: str
) -> str:

    """
    Retrieves the stored healthcare context for a specific patient from Azure Cosmos DB.

    Args:
        patient_id (str): The unique identifier of the patient, e.g. PAT-1001.

    Returns:
        str: A JSON-formatted string containing the patient's available profile,
        conditions, medications, allergies, recent vitals, and recent lab results.
    """

    try:

        patient = container.read_item(
            item=patient_id,
            partition_key=patient_id
        )

        context = {

            "patientId":
                patient["patientId"],

            "profile":
                patient.get(
                    "profile",
                    {}
                ),

            "conditions":
                patient.get(
                    "conditions",
                    []
                ),

            "medications":
                patient.get(
                    "medications",
                    []
                ),

            "allergies":
                patient.get(
                    "allergies",
                    []
                ),

            "recentVitals":
                patient.get(
                    "recentVitals",
                    {}
                ),

            "recentLabs":
                patient.get(
                    "recentLabs",
                    {}
                )
        }

        return json.dumps(
            context,
            indent=2
        )


    except Exception as e:

        return json.dumps({
            "error":
                "Patient record could not be retrieved."
        })

### Create the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Define the Function Calling Tool Capability for the Agent

In [ ]:
from azure.ai.projects.models import Tool, FunctionTool, PromptAgentDefinition

func_tool = FunctionTool(
    name="get_patient_context",
    parameters={
        "type": "object",
        "properties": {
            "patient_id": {
                "type": "string",
                "description": "The unique identifier of the patient whose healthcare context should be retrieved, e.g. PAT-1001.",
            },
        },
        "required": ["patient_id"],
        "additionalProperties": False,
    },
    description="Retrieve the stored healthcare context for a specific patient from Azure Cosmos DB.",
    strict=True,
)

### Create your Agent with Custom Function Tool

In [ ]:
instructions = """
You are a patient information assistant.

You have access to a function named
get_patient_context.

When a user asks a question about a specific
patient, use get_patient_context to retrieve
the patient's stored record.

Use the returned patient record as the
authoritative source for patient-specific facts.

Never invent patient information.

If information requested by the user is not
present in the retrieved record, state that it
is not available in the provided patient record.

Do not diagnose medical conditions.

Do not independently recommend starting,
stopping, or changing medication.

Clearly distinguish information recorded in
the patient's record from general educational
information.
"""

In [ ]:
agent_name = "health-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions=instructions,
        tools=[func_tool],
    )
)

print(f"Created agent: {agent.name} with ID: {agent.id}")

### Create the OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Create a Conversation Object for the Agent Chat System

In [ ]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")#

### Implement the Agent Loop

In [ ]:
user_query = "What medical conditions are recorded for patient PAT-1001?"

# Some more queries to try out:
# What are the latest lab results available for PAT-1001?
# Does PAT-1001 have any recorded allergies?

In [ ]:
from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam
import json

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body = {
        "agent_reference": {
            "name": agent_name,
            "type": "agent_reference"
        }
    },
    input = user_query
)

input_list: ResponseInputParam = []
# Process function calls
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_patient_context":
            # Execute the function logic for get_patient_context
            weather = get_patient_context(**json.loads(item.arguments))

            # Provide function call results to the model
            input_list.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"patient_context": weather}),
                )
            )

# Submit function results and get the final response
response = openai_client.responses.create(
    input=input_list,
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Agent response: {response.output_text}")